# unit05 レッスン: テキストの正規化と TF-IDF ベースライン

**題材** — Web から集めてきた出品テキスト(タイトル + 説明)から、**商品カテゴリ6クラスを当てる**多クラス分類。
評価指標は **macro-F1**(理由は概念4で説明する)。データは `data/` に同梱済みで、ネットワークは一切使わない。

このユニットの唯一の問いはこれだ:

> **機械学習モデルは行列しか食べられない。テキストをどうやって行列にするか。**

## このレッスンを終えると作れるようになるもの

1. Web 由来の汚れたテキスト(HTML残渣・半角カナ・全角英数・絵文字・テンプレ文言・価格の表記ゆれ)を、
   `unicodedata.normalize` と正規表現で**再現性のある1つの関数**に落とし込める
2. **janome** で日本語を形態素に分割し、品詞で絞り込んだ分かち書き文字列を作れる
3. **TF-IDF** で文書集合を疎行列に変換し、`ngram_range` / `min_df` / `sublinear_tf` の効果を語彙数とスコアで説明できる。
   **`fit` は train だけ・test は `transform` だけ**を体で覚える
4. ロジスティック回帰でベースラインを作り、**accuracy ではなく macro-F1 で読む**。
   `classification_report` を1行ずつ解釈して、少数クラスの取りこぼしを見つけられる

所要の目安: **60〜90分**。このあと演習 `ex01`〜`ex04` が続く。

## このレッスンの読み方

セルは**上から順に**実行する。構成は概念ごとに次の8ステップの繰り返し:

| 記号 | 内容 |
|---|---|
| ① | なぜこれを学ぶのか(実務のどこで使うか) |
| ② | 解説(C# との対応表・API 表) |
| ③ | **見る** — 完成コードを実行して結果を見る |
| ④ | **予測する** — 次のセルの結果を頭の中で予測する |
| ⑤ | **変えてみる** — 実行して予測と照合する |
| ⑥ | **書いてみる**(指示) |
| ⑦ | 自分で書くセル(`# ここに書く`) |
| ⑧ | チェックポイント(即時採点) |

⑦ を書かずに実行しても notebook は止まらない。⑧ が `[NG]` を出して、何が期待値なのかを教えてくれる。

In [ ]:
# ===== セットアップ: このセルを最初に1回だけ実行する =====
from pathlib import Path
import html
import re
import time
import unicodedata
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
pd.set_option("display.width", 170)
pd.set_option("display.max_columns", 30)

# データの場所。notebook をユニット直下で開いても、リポジトリのルートで開いても動くようにする。
DATA = Path("data")
if not (DATA / "train.csv").exists():
    DATA = Path("courses/kaggle-sprint/unit05-text-classical-nlp/data")
assert (DATA / "train.csv").exists(), f"train.csv が見つかりません: {DATA.resolve()}"

print("pandas:", pd.__version__, "/ numpy:", np.__version__)
print("DATA =", DATA.resolve())


# ---------- 採点ヘルパー(中身は読まなくてよい) ----------
def check(name, actual, expected, hint=""):
    import numpy as _np
    try:
        ok = actual is not None and bool(_np.all(_np.isclose(_np.asarray(actual, dtype=float), _np.asarray(expected, dtype=float))))
    except (TypeError, ValueError):
        ok = actual == expected
    if ok:
        print(f"[OK] {name}: 正解!")
    else:
        print(f"[NG] {name}: 期待値 {expected!r} / 実際 {actual!r}")
        if hint:
            print(f"     ヒント: {hint}")
    return ok


def call_safely(fn, *args, **kwargs):
    """未完成の関数を呼んでも notebook が止まらないようにするラッパ。例外なら None を返す。"""
    if not callable(fn):
        return None
    try:
        return fn(*args, **kwargs)
    except Exception as e:
        print(f"     (関数の中で例外が出ました → {type(e).__name__}: {e})")
        return None


def map_safely(fn, texts):
    """未完成の関数を文字列のリスト全体に安全に適用する。1つでも失敗したら None を返す。"""
    if not callable(fn):
        return None
    out = []
    for t in texts:
        try:
            r = fn(t)
        except Exception:
            return None
        if not isinstance(r, str):
            return None
        out.append(r)
    return out


def sp_info(X, what):
    """疎行列の rows / cols / nnz を安全に取り出す。取れなければ None(未記入でも止まらないため)。"""
    try:
        return {"rows": int(X.shape[0]), "cols": int(X.shape[1]), "nnz": int(X.nnz)}[what]
    except Exception:
        return None


print("\nセットアップ完了。ヘルパー: check / call_safely / map_safely / sp_info")

---
# 概念1 — テキストの正規化

## ① なぜ: 「ﾃﾚﾋﾞ」と「テレビ」が別の列になる日

君の会社が毎日 Web から集めてくる出品データを思い出してほしい。同じ「テレビ」が、
サイトによって `ﾃﾚﾋﾞ` `テレビ` `ＴＶ` `tv` と書かれている。説明文には `<br>` や `&nbsp;` が生で残り、
タイトルの先頭には `【送料無料】` `★即日発送★` が判で押したように付いている。

このまま機械学習に渡すと何が起きるか。モデルにとって語彙は **`Dictionary<string, int>` のキー**でしかない。
`ﾃﾚﾋﾞ` と `テレビ` は**別のキー**なので、「テレビという語がこのカテゴリを示す」という1つの手がかりが
2本、3本に**割れて薄まる**。C# で `new Dictionary<string,int>()` を `StringComparer.OrdinalIgnoreCase` 無しで作って、
`"Tokyo"` と `"tokyo"` を別々に数えてしまう事故と、まったく同じ構図だ。

しかも `【送料無料】` のような**収集サイトごとに共通するテンプレ文言**はもっと質が悪い。分類の役に立たないどころか、
「どのサイトから収集したか」を当てる手がかりになってしまい、**収集元とカテゴリに偏りがあるとリーク源になる**。
だから前処理の一発目は、いつでも「表記を揃える」「共通ノイズを落とす」の2つになる。

## ② 解説: 正規化は「別物を同じ物に潰す」作業

### Unicode 正規化(NFKC)が何をするか

`unicodedata.normalize("NFKC", s)` は Python 標準ライブラリの関数で、**見た目が違うだけの文字を1つの代表形に潰す**。
NFKC の K は「互換(compatibility)」の意味で、**互換文字を通常の文字に置き換える**のがポイント。

| 種類 | 変換前 | 変換後 |
|---|---|---|
| 半角カタカナ | `ﾃﾚﾋﾞ` | `テレビ` |
| 全角英数 | `ｉＰｈｏｎｅ` `４Ｋ` | `iPhone` `4K` |
| 全角記号・空白 | `　`(U+3000) `，` `￥` | 半角スペース `,` `¥` |
| 丸数字・組文字 | `①` `㈱` `㌢` | `1` `(株)` `センチ` |
| 濁点の分離形 | `ﾋ` + `ﾞ` | `ビ` |

C# には対応する `string.Normalize(NormalizationForm.FormKC)` がある。発想は完全に同じだ。

> **注意**: NFKC は**カタカナとひらがなは統一しない**(`テレビ` と `てれび` は別のまま)。
> 大文字小文字も揃えない。だから `.lower()` は別途かける必要がある。

### 正規表現(`re`)と C# の対応

| やりたいこと | C# | Python(`re`) |
|---|---|---|
| パターンをコンパイル | `new Regex(p)` | `re.compile(p)` |
| 含むか判定 | `Regex.IsMatch(s, p)` | `re.search(p, s)`(見つからなければ `None`) |
| 置換 | `Regex.Replace(s, p, rep)` | `re.sub(p, rep, s)` / `rx.sub(rep, s)` |
| 全部取り出す | `Regex.Matches(s, p)` | `re.findall(p, s)` |
| 大文字小文字無視 | `RegexOptions.IgnoreCase` | `re.IGNORECASE` |

**`rx.sub(置換後, 対象文字列)` の引数の順番が C# と逆**(対象が後ろ)なので、そこだけ気をつける。

### 今日使う文字列 API 一覧

| 用途 | API | 戻り値 | 注意 |
|---|---|---|---|
| HTMLエンティティを戻す | `html.unescape(s)` | str | `&amp;`→`&`、`&nbsp;`→ **改行しない空白(U+00A0)**。NFKC で普通の空白になる |
| Unicode 正規化 | `unicodedata.normalize("NFKC", s)` | str | 第1引数は形式名の文字列 |
| 小文字化 | `s.lower()` | str | C# の `ToLowerInvariant()` |
| 前後の空白削除 | `s.strip()` | str | C# の `Trim()` |
| 空白で分割 | `s.split()` | `list[str]` | 引数なしだと**連続空白をまとめて**分割する |
| 連結 | `" ".join(tokens)` | str | C# の `string.Join(" ", tokens)`。**区切り文字が主語**になるのが Python 流 |

### 正規化の順番には理由がある

1. `html.unescape` → **先にやる**。`&lt;br&gt;` のようにタグ自体がエスケープされている場合があるため
2. タグ除去 → 2番目。中身の英字がテキストに混ざる前に消す
3. **NFKC** → 3番目。ここで全角英数・半角カナが揃うので、以降のパターンが半角基準で書ける
4. 装飾記号・絵文字の除去 → NFKC 後(全角記号が半角に寄っているので)
5. テンプレ文言の除去 → 表記が揃ってから探す
6. 価格などの数値トークン化 → 表記が揃ってから
7. 小文字化 → 最後の方
8. 連続空白の畳み込み → **必ず最後**。上の各手順が空白を撒き散らすため

### そして「やりすぎ」の話

正規化は**やるほど良いものではない**。数字を全部 `<num>` に潰せば `XR-500B` という型番も
`26cm` というサイズも消え、カテゴリを見分ける強い手がかりを自分で捨てることになる。
どこまでやるかは**スコアで決める**。⑦ で「やりすぎ版」を作り、レッスンの最後に同じ条件で測って決着をつける。

In [ ]:
# GOAL: 「Webから取ってきたテキスト」がどれだけ汚れているかを、実物と件数で把握する

# STEP 1: 読み込んで shape を print する(このコースの規律。行数がどこで変わるかを常に追う)
train = pd.read_csv(DATA / "train.csv")
test = pd.read_csv(DATA / "test.csv")
print("train:", train.shape, "/ test:", test.shape)
print("列:", list(train.columns))

# STEP 2: 分類器に渡すテキストは「タイトル + 説明」を1本に連結したもの。
#          fillna("") は欠損を空文字にする(欠損のまま + すると全部 NaN になる)。
TEXTS = (train["title"].fillna("") + " " + train["description"].fillna("")).tolist()
TEXTS_TEST = (test["title"].fillna("") + " " + test["description"].fillna("")).tolist()
print("\n--- 生テキストの例(3件)---")
for t in TEXTS[:3]:
    print("  ", t[:90])

# STEP 3: 目的変数の分布。テキスト分類ではここを最初に見る(不均衡なら指標が変わる)。
print("\n--- category の分布 ---")
vc = train["category"].value_counts()
print(vc.to_string())
print("最多 / 最少 =", round(vc.max() / vc.min(), 2), "倍 → 不均衡。accuracy だけ見ていると少数クラスを見落とす")

# STEP 4: 汚れの棚卸し。re.compile(pattern) は正規表現を1回コンパイルして使い回すためのもの
#          (C# の new Regex(pattern) と同じ。re.search(p, s) は Regex.IsMatch 相当)。
PATTERNS = {
    "半角カタカナ(ﾃﾚﾋﾞ)": r"[ｦ-ﾟ]",
    "全角英数(ｉＰｈｏｎｅ)": r"[０-９Ａ-Ｚａ-ｚ]",
    "HTMLタグ(<br>)": r"<[^>]+>",
    "HTMLエンティティ(&nbsp;)": r"&[a-z]+;",
    "絵文字・装飾記号": r"[←-➿⬀-⯿\U0001F000-\U0001FAFF【】《》★◆※]",
    "テンプレ文言(送料無料 等)": r"送料無料|即日発送|匿名配送|返品不可|まとめ買い歓迎|値下げしました|最終値下げ",
    "価格の表記(円/¥/yen)": r"[¥￥]|円|yen|ｙｅｎ",
}
print("\n--- 汚れの棚卸し(4000行中の該当行数)---")
for label, pat in PATTERNS.items():
    rx = re.compile(pat, re.IGNORECASE)
    n = sum(1 for t in TEXTS if rx.search(t))
    print(f"  {label:<24} {n:>5} 行")

# STEP 5: 同じ「テレビ」が2通りの書かれ方をしている、を文字列として確認する
a, b = "ﾃﾚﾋﾞ", "テレビ"
print("\n'ﾃﾚﾋﾞ' == 'テレビ' ?", a == b, " ← Python にとっては完全に別の文字列")
print("NFKC を通すと:", unicodedata.normalize("NFKC", a) == b)
print("→ 正規化しないと、語彙表に『テレビ』の列が2本できて、手がかりが半分ずつに割れる")

## ④ 予測: 正規化を1段ずつ足すと、テキストはどう変わる?

次のセルでは、下の1件に対して正規化を**1段ずつ**適用して、その都度の状態を print する。

```
【送料無料】&nbsp;ﾃﾚﾋﾞ ４Ｋ 液晶🔥<br>お値段は¥１３，５００になります。
```

実行する前に、紙かコメントに予測を書いてみよう。

1. `html.unescape` を通した直後、`&nbsp;` は**何に**なる? 見た目で分かる?
2. **NFKC** を通した直後、この文の中で変わるのは何箇所ある?(`ﾃﾚﾋﾞ` 以外にもある)
3. 価格を `<num>` に潰す段(6番目)で、`¥１３，５００` `13,500円` `13500 円` `13500yen` の4通りは
   **本当に同じ1つのトークン**になる? どれか取りこぼしそうなものはある?
4. 最後に、`XR-500B` のような型番や `26cm` のようなサイズは、この処理を通しても生き残る?

In [ ]:
# GOAL: 正規化を1段ずつ足したときに、テキストが何にどう変わるかを目で追う

# STEP 1: 使う正規表現をまとめて用意する(re.compile は「Regex をコンパイルして使い回す」)
TAG_RE = re.compile(r"<[^>]+>")                       # <br> </b> などのHTMLタグ
DECOR_RE = re.compile(r"[【】《》★☆■◆◇▼▲▽△◎〇※♪]|[←-➿⬀-⯿️\U0001F000-\U0001FAFF]")
TEMPLATE_PHRASES = ["送料無料", "即日発送", "匿名配送", "返品不可", "まとめ買い歓迎", "値下げしました", "最終値下げ"]
TEMPLATE_RE = re.compile("|".join(TEMPLATE_PHRASES))  # | は「どれか1つ」= C# の Regex の | と同じ
PRICE_RE = re.compile(r"[¥￥]?\s*\d[\d,]*\s*(?:円|yen|jpy)|[¥￥]\s*\d[\d,]*", re.IGNORECASE)
SPACE_RE = re.compile(r"\s+")                         # 連続する空白・改行・タブ

SAMPLE = "【送料無料】&nbsp;ﾃﾚﾋﾞ ４Ｋ 液晶🔥<br>お値段は¥１３，５００になります。"

# STEP 2: 1段ずつ適用して、その都度どう変わったかを print する
s = SAMPLE
print("0. 生             :", s)
s = html.unescape(s)                      # &nbsp; &amp; などのHTMLエンティティを実体に戻す
print("1. unescape       :", s)
s = TAG_RE.sub(" ", s)                    # sub(置換後, 対象) = C# の Regex.Replace
print("2. タグ除去       :", s)
s = unicodedata.normalize("NFKC", s)      # 半角カナ→全角カナ / 全角英数→半角英数 / 全角空白→半角
print("3. NFKC           :", s)
s = DECOR_RE.sub(" ", s)                  # 絵文字・装飾記号を空白に
print("4. 装飾除去       :", s)
s = TEMPLATE_RE.sub(" ", s)               # 全文書共通のテンプレ文言を落とす
print("5. テンプレ除去   :", s)
s = PRICE_RE.sub(" <num> ", s)            # 価格の表記ゆれを1つのトークンに潰す(プレースホルダ化)
print("6. 価格→<num>    :", s)
s = s.lower()                             # C# の ToLowerInvariant()
print("7. 小文字化       :", s)
s = SPACE_RE.sub(" ", s).strip()          # 連続空白を1つに畳んで前後を削る
print("8. 空白畳み込み   :", s)


# STEP 3: これを1つの関数にまとめたものを normalize_base とする。以降のレッスンで使う。
def normalize_base(text):
    s = html.unescape(str(text))
    s = TAG_RE.sub(" ", s)
    s = unicodedata.normalize("NFKC", s)
    s = DECOR_RE.sub(" ", s)
    s = TEMPLATE_RE.sub(" ", s)
    s = PRICE_RE.sub(" <num> ", s)
    s = s.lower()
    return SPACE_RE.sub(" ", s).strip()


print("\n--- normalize_base を3件に適用 ---")
for t in TEXTS[:3]:
    print("  前:", t[:70])
    print("  後:", normalize_base(t)[:70])

# STEP 4: 4通りに書かれていた同じ価格が、1つのトークンに潰れたことを確認する
print("\n--- 価格の表記ゆれ(normalize_base 後)---")
for t in ["¥１３，５００になります", "13,500円です", "13500 円", "13500yen"]:
    print(f"  {t:<22} -> {normalize_base(t)}")
print("→ 4通りの書き方が全部 <num> という1つの列になった。桁の情報は price 列が持っているので惜しくない。")

# STEP 5: では、まだ残っている数字は? 型番・サイズ・スケールはそのまま生きている。
print("\n--- 正規化後にも残る数字 ---")
for t in ["型番 XR-500B 対応", "1/144 スケール", "26cm 厚底", "4K 液晶 39,800円"]:
    print(f"  {t:<20} -> {normalize_base(t)}")
print("→ ここをさらに潰すべきか? 潰せば語彙はもっと減るが、型番やサイズという手がかりも消える。")
print("→ 正規化は『やるほど良い』ではない。答えは ⑦ で自分の手で作って、レッスンの最後に測って決める。")

## ⑥ 書いてみる: 「やりすぎた正規化」を作って、あとで測る

`normalize_base` は価格を ` <num> ` に潰すところまでやった。**ではもう一歩踏み込んだらどうなるか。**

「数字は全部ノイズだ」という言い分にも一理ある。出品テキストの数字の多くは価格・在庫数・発送日で、
カテゴリとは無関係だ。全部潰せば語彙はさらに減り、モデルは軽くなる。

だが `1/144`(ガンプラのスケール = ホビー)、`26cm`(靴のサイズ = ファッション)、
`XR-500B`(型番 = 家電)は**強い手がかり**だ。潰せばそれも消える。

**どちらが正しいかは、測るまで分からない。** だから両方作って、レッスンの最後に同じ条件で比べる。

次のセルで `normalize_strict(text)` を書こう。

- `normalize_base(text)` を通す
- `re.sub(r"\d+", " <num> ", s)` で**残っている数字をすべて**潰す
- `SPACE_RE.sub(" ", s).strip()` で連続空白を畳んで返す

**3行**で書ける。`re.sub(パターン, 置換後, 対象)` の引数順(対象が**最後**)に注意
— C# の `Regex.Replace(対象, パターン, 置換後)` とは並びが違う。

> この関数は総合パートの比較表に**3行目**として自動で載る。書かなければその行は出ない。

In [ ]:
def normalize_strict(text):
    # ここに書く(ヒント: normalize_base(text) を通したあと、re.sub(r"\d+", " <num> ", s) で
    #                    残っている数字をすべて潰し、最後に SPACE_RE.sub(" ", s).strip() で畳む)
    return None


# 動作確認用(⑧ でも同じ文字列を使う)
S1 = "【送料無料】&nbsp;ﾃﾚﾋﾞ ４Ｋ 液晶🔥<br>お値段は¥１３，５００になります。"
S2 = "★即日発送★ ｉＰｈｏｎｅ ケース 1980yen"
S3 = "型番 XR-500B 対応 1/144 スケール 26cm"
for s in (S1, S2, S3):
    print("base  :", repr(normalize_base(s)))
    print("strict:", repr(call_safely(normalize_strict, s)))

In [ ]:
# ===== チェックポイント A: テキスト正規化 =====
_normed = map_safely(normalize_strict, TEXTS)   # 4000行に安全に適用(失敗したら None)

check("A-1 汚れの総合デパート", call_safely(normalize_strict, S1), 'テレビ <num> k 液晶 お値段は <num> になります。',
      hint="normalize_base を通してから re.sub(r\"\\d+\", \" <num> \", s) → 最後に SPACE_RE で空白を畳む。")

check("A-2 全角英数と yen 表記", call_safely(normalize_strict, S2), 'iphone ケース <num>',
      hint="NFKC も価格の <num> 化も normalize_base の中で済んでいる。上に積むだけでよい。")

check("A-3 型番・スケール・サイズが潰れる", call_safely(normalize_strict, S3), '型番 xr- <num> b 対応 <num> / <num> スケール <num> cm',
      hint="置換後の文字列は前後に半角スペースを入れた \" <num> \"。最後に SPACE_RE.sub(\" \", s).strip() で畳む。")

check("A-4 数字が残っている行数", None if _normed is None else sum(1 for t in _normed if re.search(r"\d", t)), 0,
      hint="\\d+ は『1文字以上の数字の並び』。潰し残しがあるなら re.sub の対象文字列を取り違えている。")

check("A-5 半角カタカナが残っている行数", None if _normed is None else sum(1 for t in _normed if re.search(r"[\uff66-\uff9f]", t)), 0,
      hint="normalize_base を呼び忘れている(NFKC がかかっていない)可能性が高い。")

check("A-6 正規化後の総文字数(4000行)", None if _normed is None else sum(len(t) for t in _normed), 234474,
      hint="ズレるなら置換文字列が違う。前後に空白を入れた \" <num> \" にして、最後に空白を1つに畳むこと。")

print("\n(6つとも [OK] になったら次の概念へ)")

---
# 概念2 — 日本語を単語に切る(janome)

## ① なぜ: 英語は `split()` で終わるが、日本語は終わらない

`"used sneakers 26cm".split()` — 英語はこれで単語に切れる。C# なら `s.Split(' ')` で一巻の終わりだ。
ところが `"東京都の中古テレビ"` には区切りが1つも無い。ここから `東京都 / の / 中古 / テレビ` を取り出すには、
**辞書を引きながら最も自然な区切り方を探す**専用の処理がいる。これが**形態素解析**だ。

実務では、この一手間はテキスト分類だけでなく、**検索インデックスの構築**(どの単位で転置索引を張るか)、
**名寄せ**(unit06 でやる)、**キーワード抽出**のすべての土台になる。
「日本語を扱うなら最初に通る門」だと思ってよい。

今回使う **janome** は純 Python 製の形態素解析器で、辞書のビルドも外部バイナリも不要。
完全オフラインで動くのでこの環境にちょうどいい。

## ② 解説: janome は「辞書を引いて最小コストの切り方を選ぶ」

### 仕組みを一言で

内蔵の辞書(約39万語)に対して、入力文を切る方法を全部並べ、
「単語の出やすさ + 単語のつながりやすさ」のコストが最小になる経路を選ぶ。
**競プロで言えば文字列上の最短経路DP**そのもの(ラティス上のビタビ探索)。中身は君がよく知っているアルゴリズムだ。

### API

| 用途 | API | 戻り値 | 注意 |
|---|---|---|---|
| 解析器を作る | `Tokenizer()` | Tokenizer | **辞書のロードが重い(1〜3秒)**。1個だけ作って使い回す |
| 形態素に分ける | `TOKENIZER.tokenize(text)` | Token の並び(ジェネレータ) | `for tok in ...` で回す |
| 分かち書きだけ | `TOKENIZER.tokenize(text, wakati=True)` | str の並び | 品詞を引かない分だけ速い |
| 表層形 | `tok.surface` | str | **文中に現れたそのままの形**(`買っ`) |
| 原形 | `tok.base_form` | str | 辞書形(`買う`) |
| 品詞 | `tok.part_of_speech` | str | `"名詞,一般,*,*"` のような**カンマ区切り4項目**の1本の文字列 |

`part_of_speech` が「文字列」であることに注意。大分類だけ欲しいときは `tok.part_of_speech.split(",")[0]`
と自分で切る。C# なら `POS` が enum やレコードになっていそうな場面だが、janome は素朴に文字列で返す。

> **`Tokenizer()` を関数の中で毎回作らない。** これは janome を使う人が最初に踏む地雷で、
> ループの中に書くと4000件の処理が数十分コースになる。
> C# で `new HttpClient()` をループの中で作ってはいけない、と同じ種類の話だ。
> このレッスンでは `TOKENIZER` という大文字のグローバル変数に1個だけ持つ。

### sklearn に渡す形

ベクトル化器(次の概念)が受け取るのは**「空白区切りの1本の文字列」のリスト**だ。だから最終的にこうする:

```
"東京都の中古テレビ"  →  ["東京", "都", "の", "中古", "テレビ"]  →  "東京 都 の 中古 テレビ"
```

つまり **「日本語を英語みたいな見た目にしてから」sklearn に渡す**。これで英語向けの道具がそのまま使える。

### 品詞で絞る

助詞(`の` `を` `は`)や助動詞(`です` `ます`)は、どのカテゴリの文書にも等しく出る。
語彙表を膨らませるだけで分類には効かない(TF-IDF の idf がある程度は自動で抑えてくれるが、列は減らない)。
そこで **名詞・動詞・形容詞だけ残す**フィルタをよく使う。英語圏で言う stop words 除去に相当する操作だ。

In [ ]:
# GOAL: 日本語の文が「形態素」に切られる様子と、品詞タグの中身を実物で見る

# STEP 1: janome の Tokenizer を作る。辞書(約39万語)をメモリに読むので生成が重い。
#          → プログラム全体で 1個だけ作って使い回すのが鉄則。ループの中で作ると数百倍遅くなる。
from janome.tokenizer import Tokenizer

t0 = time.time()
TOKENIZER = Tokenizer()
print(f"Tokenizer の生成: {time.time() - t0:.2f} 秒(これを4000回やったら終わらない)")

# STEP 2: tokenize(text) は「形態素オブジェクト」の並びを返す。中身を表にして見る。
sent = "東京都の中古テレビをお得に買う"
print("\n--- tokenize の中身 ---")
print(f"{'surface(表層形)':<12}{'base_form(原形)':<12}{'part_of_speech(品詞)'}")
for tok in TOKENIZER.tokenize(sent):
    print(f"{tok.surface:<14}{tok.base_form:<14}{tok.part_of_speech}")
print("\npart_of_speech はカンマ区切りの4項目。先頭が大分類(名詞/動詞/助詞/...)。")
print("split(',')[0] で大分類だけ取れる:", TOKENIZER.tokenize("走る").__iter__().__next__().part_of_speech.split(",")[0])

# STEP 3: sklearn に渡すのは「空白区切りの1本の文字列」。これを作るのが分かち書き。
#          wakati=True にすると品詞を引かずに表層形だけを返すので速い。
def wakati(text):
    return " ".join(TOKENIZER.tokenize(str(text), wakati=True))


print("\n--- 分かち書きの結果 ---")
print("入力:", sent)
print("出力:", wakati(sent))
print("\n英語なら 'used sneakers 26cm'.split() で済む。日本語は区切りが無いのでこの一手間が要る。")

## ④ 予測: 正規化してから切るか、切ってから困るか

次のセルでは、同じ内容の文を **正規化なし** と **`normalize_base` 済み** の2通りで分かち書きして比べる。

```
ﾃﾚﾋﾞ ４Ｋ 液晶🔥 33,800円
```

実行する前に予測してみよう。

1. 正規化していない `ﾃﾚﾋﾞ` を janome に渡すと、辞書に載っていない語はどう切られる? 1つのまま? バラバラ?
2. 200件ぶんの**異なりトークン数**(語彙の大きさ)は、正規化ありと無しでどちらが大きい? だいたい何割違う?
3. 正規化で「消えた」トークンには、どんなものが並ぶと思う?
4. `この中古のテレビはとても状態が良いです` から**名詞・動詞・形容詞だけ**残すと、何個のトークンが残る?

In [ ]:
# GOAL: 正規化してから分かち書きすると、切れ方と語彙がどう変わるかを比べる

dirty = "ﾃﾚﾋﾞ ４Ｋ 液晶🔥 33,800円"

print("--- 正規化なしで分かち書き ---")
print(wakati(dirty))
print("--- normalize_base のあとで分かち書き ---")
print(wakati(normalize_base(dirty)))
print("\n上下で同じ意味の語が別トークンになっているのが分かる(ﾃﾚﾋﾞ / テレビ)。")

# STEP 2: コーパス全体ではどれだけ語彙が違うか。小さめの200件で比べる(全件は後でやる)。
sub = TEXTS[:200]
vocab_raw = set(w for t in sub for w in wakati(t).split())
vocab_norm = set(w for t in sub for w in wakati(normalize_base(t)).split())
print(f"\n200件の異なりトークン数: 正規化なし {len(vocab_raw)} / 正規化あり {len(vocab_norm)}")
print("正規化で消えたトークンの例:", sorted(vocab_raw - vocab_norm)[:12])

# STEP 3: 品詞で絞ると何が落ちるか。助詞・助動詞(の/を/に/です)は分類にほとんど寄与しない。
sent2 = "この中古のテレビはとても状態が良いです"
print("\n--- 品詞フィルタの下ごしらえ ---")
for tok in TOKENIZER.tokenize(sent2):
    print(f"  {tok.surface:<6} {tok.part_of_speech.split(',')[0]}")
print("全部:", wakati(sent2))
print("→ 名詞・動詞・形容詞だけ残すとどうなるか、を ⑦ で書く。")

## ⑥ 書いてみる: 品詞で絞った分かち書き

③ で作った `wakati(text)` は全トークンを残す版だった。
今度は **名詞・動詞・形容詞だけ**を残す `wakati_content(text)` を書こう。

- `TOKENIZER.tokenize(text)` を `for` で回す(`wakati=True` は**付けない**。品詞が要るので)
- `tok.part_of_speech.split(",")[0]` が `"名詞"` `"動詞"` `"形容詞"` のいずれかのものだけ採用する
- `tok.surface` を集めて `" ".join(...)` で1本の文字列にして返す

**2〜4行**で書ける。リスト内包表記(`[x for x in xs if 条件]`)を使うと1式で書ける
— TypeScript の `xs.filter(...).map(...)` を1つにまとめたものだと思えばよい。

> このあとの本編では、比較を単純にするため品詞フィルタ**なし**の `wakati` を使う。
> 「品詞で絞ると精度が上がるのか」は演習 `ex02` で自分で測ることになる。

In [ ]:
def wakati_content(text):
    # ここに書く(ヒント: TOKENIZER.tokenize(text) を回し、tok.part_of_speech.split(",")[0] が
    #                    名詞 / 動詞 / 形容詞 のものだけ tok.surface を集めて " ".join する)
    return None


T1 = "この中古のテレビはとても状態が良いです"
T2 = "送料は無料で発送します"
print(repr(call_safely(wakati_content, T1)))
print(repr(call_safely(wakati_content, T2)))

In [ ]:
# ===== チェックポイント B: janome の分かち書きと品詞フィルタ =====
check("B-1 助詞・助動詞が落ちているか", call_safely(wakati_content, T1), '中古 テレビ 状態 良い',
      hint="tok.part_of_speech は '名詞,一般,*,*' のような文字列。split(\",\")[0] が大分類。")

check("B-2 トークン数", None if not isinstance(call_safely(wakati_content, T1), str) else len(call_safely(wakati_content, T1).split()), 4,
      hint="区切りは半角スペース1つ。\" \".join(...) で連結する。")

check("B-3 別の文でも同じ規則で動くか", call_safely(wakati_content, T2), '送料 無料 発送 し',
      hint="残すのは 名詞 / 動詞 / 形容詞 の3つだけ。副詞・助詞・記号は落とす。")

print("\n(3つとも [OK] になったら次の概念へ)")

### ここでコーパスを1回だけ作る

概念1の `normalize_base` と概念2の `wakati` を組み合わせて、
**以降ずっと使う「正規化 → 分かち書き済み」のテキスト**を作っておく。

janome は 5000 件で十数秒かかる。**同じ計算を何度もしない**のは実務でもコンペでも基本で、
「重い前処理は1回だけ実行して変数(あるいはファイル)に置く」が鉄則になる。

In [ ]:
# GOAL: 以降ずっと使う「正規化 → 分かち書き済み」のコーパスを1回だけ作って変数に置く

t0 = time.time()
DOCS_TRAIN = [wakati(normalize_base(t)) for t in TEXTS]        # 4000件
DOCS_TEST = [wakati(normalize_base(t)) for t in TEXTS_TEST]    # 1000件
print(f"分かち書き完了: {len(DOCS_TRAIN)} + {len(DOCS_TEST)} 件 / {time.time() - t0:.1f} 秒")
print("\n--- 出来上がり(2件)---")
for d in DOCS_TRAIN[:2]:
    print("  ", d[:100])
print("\nこの形(空白区切りの1行1文書)が sklearn のベクトル化器の入力になる。")

---
# 概念3 — BoW と TF-IDF: テキストを行列にする

## ① なぜ: モデルは行列しか食べられない

ロジスティック回帰も LightGBM もニューラルネットも、入力は例外なく**数値の行列**だ。
だから「文章を数値の並びにする」ステップが必ず要る。ここの設計がテキスト分類の勝敗の大半を決める。

実務での顔ぶれも同じで、**検索エンジンのランキング**(TF-IDF は検索の古典そのもの)、
**類似文書の検出**(重複記事のクラスタリング)、**問い合わせの自動振り分け**。
どれも「文書 → ベクトル」の作り方を選ぶところから始まる。

## ② 解説: BoW は語彙表 + カウント、TF-IDF はそれに重み付け

### Bag of Words(BoW)

「文書を、語の出現回数だけで表す。語順は捨てる」という割り切り。C# で書けばこうだ:

```csharp
// 語彙表: 語 -> 列番号
var vocab = new Dictionary<string, int>();
// 各文書は 疎ベクトル(列番号 -> 回数)
var doc = new Dictionary<int, double>();
```

**語彙表 `Dictionary<string,int>` が列の定義そのもの**である、という点だけ掴めば BoW は理解できたも同然だ。
文書数 4000・語彙 4000 なら 4000×4000 の行列になるが、1文書に出る語はせいぜい数十なので、
**中身の 99% 以上はゼロ**。だから `Dictionary<int,double>` のような**疎な**持ち方をする。

### TF-IDF

BoW の素のカウントには問題がある。`する` `なる` のような語はどの文書にも大量に出るのに、
カテゴリを見分ける役には立たない。そこで **「その文書での多さ(TF)」×「全体での珍しさ(IDF)」** で重み付けする。

$$\mathrm{tfidf}(t, d) = \mathrm{tf}(t, d) \times \mathrm{idf}(t), \qquad
  \mathrm{idf}(t) = \ln\frac{1 + n}{1 + \mathrm{df}(t)} + 1$$

- $\mathrm{tf}(t,d)$: 文書 $d$ における語 $t$ の出現回数
- $\mathrm{df}(t)$: 語 $t$ が出現した**文書の数**
- $n$: 全文書数

全文書に出る語は $\mathrm{df} = n$ なので idf が最小(≈1)になり、重みが自動的に潰れる。
**「送料無料を消し忘れても TF-IDF がある程度は面倒を見てくれる」**わけだが、
それでも列は消費するし、偏りがあれば手がかりとして使われてしまうので、前処理で落とす方が確実だ。

最後に各行は **L2 正規化**(長さ1のベクトルに)される。長い文書ほど全部の値が大きくなる不公平を消すため。

### 主なパラメータ

| パラメータ | 意味 | 効果 |
|---|---|---|
| `ngram_range=(1, 2)` | 1語と「連続2語」の両方を列にする | `まとめ 売り` のような並びを捉えられる。**語彙数は数倍に膨らむ** |
| `min_df=2` | **2文書未満**にしか出ない語を捨てる | 1回しか出ない語は学習に使えない。語彙が大きく減る |
| `max_df=0.9` | 90%超の文書に出る語を捨てる | テンプレ文言の自動除去に使える |
| `max_features=20000` | 頻度上位 N 語だけ残す | メモリの上限を切りたいとき |
| `sublinear_tf=True` | tf を $1 + \log(\mathrm{tf})$ にする | 「10回出た語が1回の10倍重要」ではない、という現実に寄せる |
| `token_pattern` | 語とみなす正規表現 | **既定は `(?u)\b\w\w+\b` = 2文字以上**。日本語の1文字語(`本`/`服`)が落ちる罠 |

### 出力は scipy の疎行列(CSR)

`fit_transform` が返すのは NumPy 配列ではなく **`scipy.sparse.csr_matrix`**。
非ゼロの値だけを (行, 列, 値) の形で持つデータ構造で、`.shape` は `(文書数, 語彙数)`、
`.nnz` が**非ゼロ要素の個数**。sklearn の線形モデルは疎行列をそのまま受け取れる。

> **`.toarray()` を軽い気持ちで呼ばない。** 4000 × 4000 の密行列は float64 で 128MB。
> 語彙が10万語になれば 3GB を超えて即死する。中身を見たいときは小さなサンプルだけにする。

### 最重要: `fit_transform` と `transform` の違い

| メソッド | やること | 使ってよい対象 |
|---|---|---|
| `fit(X)` | **語彙表と idf を決める** | train のみ |
| `transform(X)` | 決まった語彙表で行列に変換する | train / test / 本番データ |
| `fit_transform(X)` | 上2つを一度に | **train のみ** |

test に `fit_transform` を使うと、(a) 列数と列の意味が train と食い違ってモデルに渡せない、
(b) test の分布情報が重みに混ざる = **語彙リーク**、の2つが同時に起きる。
C# のアナロジーで言えば、**`fit` は状態を持つ変換器の初期化、`transform` はその状態を使った純粋関数**。
初期化を2回やってはいけない。

In [ ]:
# GOAL: 「文書の集合 → 数値の行列」への変換を、5文だけの小さなコーパスで完全に目で追う

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

demo = [
    "テレビ 液晶 中古",
    "テレビ リモコン 中古",
    "スニーカー 26cm 中古",
    "スニーカー 新品",
    "中古 中古 中古 ギター",
]

# STEP 1: CountVectorizer = 語彙表を作って「各語が何回出たか」を数えるだけの変換器。
#          fit_transform(docs) は「語彙表を作る(fit)」+「行列に変換する(transform)」を一度にやる。
cvec = CountVectorizer()
Xc = cvec.fit_transform(demo)
vocab = cvec.get_feature_names_out()      # 列番号 -> 語 の対応表(語彙表)
print("語彙:", list(vocab))
print("shape (文書数, 語彙数):", Xc.shape)
print("\n--- カウント行列(BoW)---")
print(pd.DataFrame(Xc.toarray(), columns=vocab))
print("\n※ 既定の token_pattern は『2文字以上の語』しか拾わない。日本語の1文字語(本/服)は落ちる。")

# STEP 2: TfidfVectorizer = 同じ語彙表を作りつつ、値を TF-IDF の重みにする。
#          idf = log((1+n) / (1+df)) + 1。df は「その語が出た文書数」。全文書に出る語ほど idf が小さい。
tvec = TfidfVectorizer()
Xt = tvec.fit_transform(demo)
print("\n--- TF-IDF 行列(行ごとに長さ1へ正規化されている)---")
print(pd.DataFrame(np.round(Xt.toarray(), 3), columns=tvec.get_feature_names_out()))
print("\nidf(語ごとの珍しさ):")
print(pd.Series(np.round(tvec.idf_, 3), index=tvec.get_feature_names_out()).to_string())
print("→ 5文書中4文書に出る『中古』は idf が最小 = ほとんど手がかりにならない、と自動で判断されている")

# STEP 3: fit は train だけ。test には transform だけを使う。
new_docs = ["テレビ 新品 保証", "ギター ケース"]
Xnew = tvec.transform(new_docs)           # ← fit_transform ではない
print("\ntest 側 shape:", Xnew.shape, "← 列数は train の語彙数のまま。未知語(保証/ケース)は捨てられる")

# STEP 4: 本番コーパスをベクトル化する。ここで作った TFIDF / X_all を以降で使う。
TFIDF = TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True)
X_all = TFIDF.fit_transform(DOCS_TRAIN)
print("\n--- 本番コーパス(4000文書)---")
print("shape:", X_all.shape, "/ 非ゼロ要素 nnz:", X_all.nnz)
print(f"密度: {X_all.nnz / (X_all.shape[0] * X_all.shape[1]) * 100:.2f}%  ← 99%以上がゼロ")
print(f"toarray() で潰すと {X_all.shape[0] * X_all.shape[1] * 8 / 1e6:.0f} MB。疎行列のまま渡すこと。")

## ④ 予測: パラメータを動かすと語彙数はどう動く?

次のセルでは、本番コーパス(4000文書)に対して5通りの設定でベクトル化して、
**語彙数・非ゼロ要素数(nnz)・密度**を表にする。

| 設定 | ngram_range | min_df |
|---|---|---|
| A | (1, 1) | 1 |
| B | (1, 1) | 2 |
| C | (1, 2) | 1 |
| D | (1, 2) | 2 |
| E | (1, 2) | 5 |

実行する前に予測しよう。

1. A → B(`min_df` を 1→2)で語彙数は何割くらい減る? 半分? 1割?
2. A → C(bigram を足す)で語彙数は何倍になる? 2倍? 10倍?
3. **密度**(非ゼロ要素の割合)は、語彙が増えると上がる? 下がる?
4. `test` に `fit_transform` してしまった場合の列数は、train の語彙数より多い? 少ない?

In [ ]:
# GOAL: ngram_range / min_df / max_features を動かして、語彙数と行列サイズのトレードオフを見る

settings = [
    ("ngram(1,1), min_df=1", dict(ngram_range=(1, 1), min_df=1)),
    ("ngram(1,1), min_df=2", dict(ngram_range=(1, 1), min_df=2)),
    ("ngram(1,2), min_df=1", dict(ngram_range=(1, 2), min_df=1)),
    ("ngram(1,2), min_df=2", dict(ngram_range=(1, 2), min_df=2)),
    ("ngram(1,2), min_df=5", dict(ngram_range=(1, 2), min_df=5)),
]
print(f"{'設定':<26}{'語彙数':>8}{'nnz':>10}{'密度%':>9}")
print("-" * 53)
for label, kw in settings:
    v = TfidfVectorizer(sublinear_tf=True, **kw)
    X = v.fit_transform(DOCS_TRAIN)
    print(f"{label:<24}{X.shape[1]:>8}{X.nnz:>10}{X.nnz / (X.shape[0] * X.shape[1]) * 100:>9.2f}")

# STEP 2: bigram が何を拾っているのかを実物で見る
v2 = TfidfVectorizer(ngram_range=(2, 2), min_df=5)
v2.fit(DOCS_TRAIN)
print("\nbigram の例:", list(v2.get_feature_names_out())[:10])
print("→ 単語単体では分からない『まとめ 売り』『本体 のみ』のような並びを列にできる")

# STEP 3: sublinear_tf の効き方。tf をそのまま使うか 1+log(tf) にするか。
tf = np.array([1, 2, 5, 10, 50])
print("\ntf        :", tf)
print("1+log(tf) :", np.round(1 + np.log(tf), 3), "← 10回出た語を10倍重くはしない(冪等に近づける)")

# STEP 4: やってはいけない例 —— test に fit_transform してしまうと何が起きるか
bad = TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True)
X_bad = bad.fit_transform(DOCS_TEST)
print("\ntest に fit_transform した場合の shape:", X_bad.shape)
print("train 側の語彙数:", X_all.shape[1], "→ 列数も列の意味も違う。学習済みモデルには渡せない。")
print("さらに test の情報(どの語が何文書に出るか)が重みに混ざる = 語彙リーク。")

## ⑥ 書いてみる: train で fit、test は transform だけ

このレッスンの標準設定で、train と test の両方を行列にしよう。

```
TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True)
```

- `vec` … 上の設定で作った `TfidfVectorizer`
- `X_tr` … `DOCS_TRAIN`(4000件)を変換した行列
- `X_te` … `DOCS_TEST`(1000件)を変換した行列

**3行**で書ける。ポイントはただ1つ、**`fit` してよいのは train だけ**。
`X_te` の列数が `X_tr` と一致していれば正しく書けている(⑧ が確かめてくれる)。

In [ ]:
vec = None    # TfidfVectorizer 本体
X_tr = None   # train 側の行列
X_te = None   # test 側の行列

# ここに書く(ヒント: fit してよいのは DOCS_TRAIN だけ。DOCS_TEST には transform だけを使う)


print("vec :", vec)
print("X_tr:", None if X_tr is None else X_tr.shape)
print("X_te:", None if X_te is None else X_te.shape)

In [ ]:
# ===== チェックポイント C: TF-IDF の fit / transform =====
check("C-1 train 側の語彙数", sp_info(X_tr, "cols"), 1524,
      hint="TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True) の3つを全部指定する。")

check("C-2 train 側の行数", sp_info(X_tr, "rows"), 4000,
      hint="fit_transform に渡すのは DOCS_TRAIN(4000件)。")

check("C-3 test 側の行数", sp_info(X_te, "rows"), 1000,
      hint="transform に渡すのは DOCS_TEST(1000件)。")

check("C-4 test 側の列数(= train と同じ語彙)", sp_info(X_te, "cols"), 1524,
      hint="test に fit_transform を使うと列数が変わる。test は transform だけ。")

check("C-5 train 側の非ゼロ要素数", sp_info(X_tr, "nnz"), 105341,
      hint="ズレるなら min_df か ngram_range か sublinear_tf のどれかが既定値のまま。")

print("\n(5つとも [OK] になったら次の概念へ)")

---
# 概念4 — 線形モデルのベースラインと、その読み方

## ① なぜ: ベースラインは「比較の原点」であって、ゴールではない

TF-IDF + 線形モデルは、テキスト分類における **「まずこれを置け」の定番**だ。
数秒で回り、中身が読め(どの語がどのクラスを押しているか係数で分かる)、そこそこ強い。
ここを原点にして初めて「BERT を使う価値があるか」を判断できる。
**原点を置かずに大きなモデルから始めるのは、目盛りの無い定規で測るのと同じ**だ。

そしてもう1つ大事なのが**スコアの読み方**。今日のデータは最多クラスと最少クラスで **4.3倍**の開きがある。
こういうときに accuracy だけを見ると、**少数クラスを丸ごと捨てたモデルが「良いモデル」に見える**。
実務でこれをやると「不正検知の精度99%です(1件も検知していない)」という笑えない報告書が出来上がる。

## ② 解説: ロジスティック回帰と、指標の選び方

### 多クラスのロジスティック回帰

各クラス $k$ に対して重みベクトル $w_k$ を持ち、スコア $w_k \cdot x$ を softmax で確率に変える。
**クラス数 × 語彙数 の係数行列**を学習する、と思えばよい。

| パラメータ | 意味 |
|---|---|
| `C` | 正則化の強さの**逆数**。大きいほど正則化が弱い(訓練データに寄る)。既定は 1.0、今回は 4.0 |
| `max_iter` | 最適化の反復上限。テキストは次元が高く収束が遅いので 2000 にしておく(足りないと警告が出る) |
| `class_weight="balanced"` | クラスの重みを $\frac{n}{K \cdot n_k}$ にする = **少数クラスの誤りを重く罰する** |

| 用途 | API | 戻り値 |
|---|---|---|
| 学習 | `clf.fit(X, y)` | 自分自身 |
| ラベル予測 | `clf.predict(X)` | `(n,)` のラベル配列 |
| 確率予測 | `clf.predict_proba(X)` | `(n, クラス数)`。列の順は `clf.classes_` |
| 係数 | `clf.coef_` | `(クラス数, 語彙数)` |

> 兄弟分として `LinearSVC`(速くて強いが `predict_proba` が**無い**。代わりに `decision_function`)、
> `SGDClassifier`(データが巨大なとき)、`ComplementNB`(不均衡テキストに強い素朴ベイズ)がある。
> どれも同じ `fit` / `predict` のインターフェースを持つ — **C# のインターフェース実装と同じで、差し替えが効く**。

### Pipeline: 前処理とモデルを1つの部品にする

`make_pipeline(vectorizer, classifier)` は「ベクトル化 → 分類」を**1つのモデルのように振る舞う部品**にまとめる。
これが効くのは交差検証のときで、**fold ごとに『学習側だけで fit → 検証側は transform』が自動で守られる**。
自分で `fit_transform` を書くと、うっかり全データで fit してリークさせる事故が起きる。Pipeline はそれを構造的に防ぐ。

### 指標: accuracy / micro-F1 / macro-F1

| 指標 | 定義 | 性格 |
|---|---|---|
| accuracy | 正解した行の割合 | **多数クラスの成績にほぼ支配される** |
| micro-F1 | 全行を一括で集計した F1 | 多クラス単一ラベルでは **accuracy と一致する** |
| **macro-F1** | クラスごとに F1 を出して**単純平均** | **どのクラスも1票**。少数クラスの失敗が正面から効く |

今日のデータは 家電 1053 件に対し ベビー・キッズ 245 件。
ベビー・キッズを全部落としても accuracy は 6% しか下がらないが、macro-F1 は 1/6 が丸ごと消えて大打撃を受ける。
**だからこのコンペは macro-F1 で評価する。**

### `classification_report` の読み方

```
                precision  recall  f1-score  support
ベビー・キッズ      0.94     0.78      0.85      245
```

- **precision**(適合率) = そのクラスと**予測した**うち本当に正解だった割合 → 誤検知の少なさ
- **recall**(再現率) = 本当にそのクラスだった行のうち**拾えた**割合 → 取りこぼしの少なさ
- **f1** = precision と recall の調和平均(どちらかが低いと大きく下がる)
- **support** = そのクラスの実際の件数。**ここが小さい行の数字は揺れやすい**ので過信しない

`precision` が高く `recall` が低い、は「自信のある少数だけ拾っている」状態。
不均衡データで少数クラスに起きる典型的な症状で、`class_weight="balanced"` はここに効く。

In [ ]:
# GOAL: 線形モデルのベースラインを正しい手順(CV)で測り、accuracy と macro-F1 の差を見る

from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import accuracy_score, f1_score, classification_report

y = train["category"].to_numpy()

# STEP 1: 「ベクトル化 → 分類」を1本の Pipeline にする。
#          こうすると fold ごとに『学習側だけで fit』が自動で守られる(語彙リークの防止)。
#          C# で言えば処理の連結を1つのインターフェースに包む DI 的な差し替え可能パイプライン。
def make_model(**lr_kw):
    return make_pipeline(
        TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True),
        LogisticRegression(C=4.0, max_iter=2000, **lr_kw),
    )


# STEP 2: 5分割の交差検証で、全行に対する「学習に使っていない状態での予測」(OOF)を作る。
#          StratifiedKFold は各 fold のクラス比を元の比率に保つ分割(不均衡データでは必須)。
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
t0 = time.time()
oof_pred = cross_val_predict(make_model(), DOCS_TRAIN, y, cv=cv)
print(f"CV 完了: {time.time() - t0:.1f} 秒")

# STEP 3: 2つの指標を並べる。
acc = accuracy_score(y, oof_pred)
macro = f1_score(y, oof_pred, average="macro")
micro = f1_score(y, oof_pred, average="micro")
print(f"\naccuracy : {acc:.4f}")
print(f"micro-F1 : {micro:.4f}  ← 多クラス単一ラベルでは accuracy と一致する")
print(f"macro-F1 : {macro:.4f}  ← クラスごとの F1 の単純平均。少数クラスも1票")

# STEP 4: classification_report はクラスごとの成績表。
print("\n--- classification_report ---")
print(classification_report(y, oof_pred, digits=3, zero_division=0))
print("読み方(1行ずつ):")
print("  precision = そのクラスと予測したうちの正解率(誤検知の少なさ)")
print("  recall    = 実際にそのクラスの行のうち拾えた割合(取りこぼしの少なさ)")
print("  f1        = precision と recall の調和平均")
print("  support   = 実際の件数。ここが小さいクラスの数字は揺れやすい")
print("\n注目: support 245 の『ベビー・キッズ』の recall。全体 accuracy には 6% しか影響しないので、")
print("      accuracy だけ見ていると『そこそこ良いモデル』に見えてしまう。macro-F1 はここを拾う。")

## ④ 予測: `class_weight="balanced"` は何を上げ、何を下げる?

次のセルでは `class_weight="balanced"` を付けて、まったく同じ条件でもう一度 CV する。

実行する前に予測しよう。

1. **accuracy** は上がる? 下がる?
2. **macro-F1** は上がる? 下がる?
3. **ベビー・キッズ の recall** は上がる? 下がる? そのとき precision はどうなる?
4. さらにセルの後半では、クラスごとに係数が大きい語 top8 を並べる。
   `本・音楽` のトップに来る語を3つ挙げてみよう。もし `送料` `無料` のような語が並んでいたら、それは何を意味する?

In [ ]:
# GOAL: class_weight='balanced' が accuracy と macro-F1 に与える効果を測り、モデルの手がかりを覗く

# STEP 1: class_weight='balanced' は「件数の少ないクラスの誤りを重く罰する」重み付け。
#          重み = 全体件数 / (クラス数 * そのクラスの件数)。
t0 = time.time()
oof_bal = cross_val_predict(make_model(class_weight="balanced"), DOCS_TRAIN, y, cv=cv)
print(f"CV 完了: {time.time() - t0:.1f} 秒\n")

print(f"{'設定':<22}{'accuracy':>10}{'macro-F1':>10}{'ベビーrecall':>14}")
print("-" * 56)
for label, p in [("既定", oof_pred), ("balanced", oof_bal)]:
    r = classification_report(y, p, output_dict=True, zero_division=0)
    print(f"{label:<20}{accuracy_score(y, p):>10.4f}{f1_score(y, p, average='macro'):>10.4f}"
          f"{r['ベビー・キッズ']['recall']:>14.3f}")

# STEP 2: 係数の大きい語を見る。線形モデルは coef_ が (クラス数, 語彙数) の行列で、
#          値が大きい語ほど「そのクラスらしさ」を押し上げる。中身が読めるのが線形モデルの利点。
clf_full = LogisticRegression(C=4.0, max_iter=2000).fit(X_all, y)
feats = np.array(TFIDF.get_feature_names_out())
print("\ncoef_ の shape:", clf_full.coef_.shape, "= (クラス数, 語彙数)")
print("\n--- クラスごとの効いている語 top8 ---")
for i, cls in enumerate(clf_full.classes_):
    top = feats[np.argsort(clf_full.coef_[i])[::-1][:8]]
    print(f"  {cls:<10} {' / '.join(top)}")
print("\n→ ここに『送料無料』のようなテンプレ語が並んでいたら、正規化で落とし損ねているサイン。")

## ⑥ 書いてみる: レポートから数字を取り出す

スコアは「眺めるもの」ではなく「**取り出して比較するもの**」だ。
実務では毎回この3つを辞書やログに落として、実験の履歴として残す。

③ で作った `y` と `oof_pred` を使って(**学習し直す必要はない**)、次の3つの変数に値を入れよう。

| 変数 | 中身 |
|---|---|
| `d_acc` | accuracy |
| `d_macro_f1` | macro-F1 |
| `d_baby_recall` | `ベビー・キッズ` クラスの recall |

使う道具:

- `accuracy_score(y_true, y_pred)`
- `f1_score(y_true, y_pred, average="macro")` — **`average` を省くと多クラスではエラーになる**
- `classification_report(y_true, y_pred, output_dict=True)` — レポートを**辞書の入れ子**で返す。
  `d["クラス名"]["recall"]` のように取り出せる(Python の dict は C# の `Dictionary<string, object>` 相当)

**3〜4行**で書ける。

In [ ]:
# ③ で作った y と oof_pred を使う。新しく学習し直す必要はない。
d_acc = None          # accuracy
d_macro_f1 = None     # macro-F1
d_baby_recall = None  # 「ベビー・キッズ」の recall

# ここに書く(ヒント: classification_report(..., output_dict=True) は dict を返す。
#                    dict["ベビー・キッズ"]["recall"] のように取り出せる)


print("accuracy      :", d_acc)
print("macro-F1      :", d_macro_f1)
print("ベビー recall :", d_baby_recall)

In [ ]:
# ===== チェックポイント D: 指標の読み取り =====
check("D-1 accuracy", d_acc, 0.9395,
      hint="accuracy_score(y, oof_pred)。classification_report の dict なら [\"accuracy\"] でも取れる。")

check("D-2 macro-F1", d_macro_f1, 0.9383971231402984,
      hint="f1_score(y, oof_pred, average=\"macro\")。average を付け忘れると多クラスでエラーになる。")

check("D-3 ベビー・キッズ の recall", d_baby_recall, 0.8693877551020408,
      hint="classification_report(y, oof_pred, output_dict=True) の戻り値は dict の入れ子。キーはクラス名の文字列。")

print("\n(3つとも [OK] になったら総合パートへ)")

---
# 総合 — 正規化は本当に効いたのか

ここまで「正規化は大事だ」と言葉で言ってきた。だが前処理の良し悪しは**指標でしか決まらない**。
言葉で正しそうな処理がスコアを下げることは日常的に起きる(さっきの「型番を潰す」話がまさにそれだ)。

次のセルは、**入力テキスト以外をすべて固定**して3条件を比べる:

| | テキスト | ベクトル化 | モデル | 分割 |
|---|---|---|---|---|
| ① | 生のまま分かち書き | TF-IDF (1,2) / min_df=2 / sublinear | LogisticRegression(C=4.0) | StratifiedKFold(5, seed=0) |
| ② | `normalize_base` を通してから | 同上 | 同上 | 同上 |
| ③ | `normalize_strict`(⑦ の やりすぎ版) | 同上 | 同上 | 同上 |

**変えているのは入力テキストだけ。** これが前処理の効果を測るときの唯一正しい形だ。
ベクトル化やモデルを同時に変えると、スコアが動いた理由が分からなくなる。

見るべきは2つの数字が**同時に**どう動くかだ。

- **語彙数**: 正規化で表記ゆれが潰れるぶん減るはず。どれくらい?
- **macro-F1**: 語彙(= 列数)が減ったのに、スコアは上がるのか下がるのか?

普通、特徴量を削ればスコアは下がる。①→② でそれが**上がる**なら、
削られたのは情報ではなく**ノイズと重複**だったという証拠になる。
そして ②→③ でスコアが**下がる**なら、そこが「やりすぎの境界線」だ。

In [ ]:
# GOAL: このレッスンの核心 —— 正規化は本当に効いたのか、を同じ条件で測って決着させる

# STEP 1: 正規化「なし」のコーパスを作る(生テキストをそのまま分かち書きするだけ)
t0 = time.time()
RAWDOCS_TRAIN = [wakati(t) for t in TEXTS]
RAWDOCS_TEST = [wakati(t) for t in TEXTS_TEST]
print(f"生テキストの分かち書き: {time.time() - t0:.1f} 秒")

# STEP 2: 語彙数を比べる(train 全体で fit したときの列数)
v_raw = TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True)
X_raw = v_raw.fit_transform(RAWDOCS_TRAIN)
print(f"\n語彙数: 正規化なし {X_raw.shape[1]} / 正規化あり {X_all.shape[1]}")

# STEP 3: まったく同じモデル・同じ分割で CV する。違うのは入力テキストだけ。
t0 = time.time()
oof_raw = cross_val_predict(make_model(), RAWDOCS_TRAIN, y, cv=cv)
print(f"CV 完了: {time.time() - t0:.1f} 秒")

rows = [
    ("① 素のテキスト", X_raw.shape[1], oof_raw),
    ("② 正規化してから", X_all.shape[1], oof_pred),
]

# STEP 4: ⑦ で書いた「やりすぎ版」も、まったく同じ条件で測る
_strict_texts = map_safely(globals().get("normalize_strict"), TEXTS)
if _strict_texts is None:
    print("\n(概念1の ⑦ が未記入なので ③ はスキップ。書いてから再実行しよう)")
else:
    DOCS_STRICT = [wakati(t) for t in _strict_texts]
    X_st = TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True).fit_transform(DOCS_STRICT)
    oof_st = cross_val_predict(make_model(), DOCS_STRICT, y, cv=cv)
    rows.append(("③ 数字を全部潰す(⑦)", X_st.shape[1], oof_st))

print(f"\n{'前処理':<22}{'語彙数':>8}{'accuracy':>11}{'macro-F1':>11}")
print("-" * 52)
for label, nvocab, p in rows:
    print(f"{label:<20}{nvocab:>8}{accuracy_score(y, p):>11.4f}{f1_score(y, p, average='macro'):>11.4f}")
print("\n① → ② で語彙は大きく減り、それでいてスコアは上がった。")
print("『表記ゆれで割れていた同じ手がかりが1本にまとまった』ことの直接の証拠。")
print("② → ③ はどうか? 語彙は期待通り減った? スコアは? そこが『やりすぎ』の境界線だ。")

## 結論

| 前処理 | 語彙数 | accuracy | macro-F1 |
|---|---|---|---|
| ① 素のテキスト | 4185 | 0.9243 | 0.9219 |
| ② 正規化してから | 1524 | 0.9395 | 0.9384 |
| ③ さらに数字を全部潰す(⑦) | 1530 | 0.9390 | 0.9381 |

### ① → ②(正規化は効いた)

**語彙は 4185 → 1524(2661 語減って 36%)。それでいて macro-F1 は 0.9219 → 0.9384 に上がった。**

これがこのユニットの核心だ。減ったのは情報ではなく、**表記ゆれによって水増しされていた重複の列**だった。
`ﾃﾚﾋﾞ` と `テレビ` が2本の列に割れていたのを1本に束ねたので、
1つ1つの列に載る証拠の量が増え、少ないデータでも係数が安定して推定できるようになった。

**副作用として推論も速くメモリも小さくなる。**
「精度」と「速度」がトレードオフにならない改善は珍しく、前処理はその数少ない場所のひとつだ。

### ② → ③(やりすぎると 下がった)

**狙いは「語彙をさらに減らす」ことだった。結果は語彙 1524 → 1530 でむしろ増えた。**
そして macro-F1 は 0.9384 → 0.9381 と下がった。**得たものが何も無い。**

なぜ語彙が減らなかったのか。⑦ の出力をもう一度見てほしい:

```
'型番 XR-500B 対応 1/144 スケール 26cm'
  → '型番 xr- <num> b 対応 <num> / <num> スケール <num> cm'
```

`xr-500b` という **1トークンだった型番が `xr-` / `<num>` / `b` に割れて**、
分かち書き後には新しいトークンと新しい bigram を生んでしまった。
潰したはずが**増やしていた**わけだ。これは正規化を設計するときに実際によく起きる失敗で、
**「置換した結果が次の工程(分かち書き)でどう切られるか」まで見ないと判断できない**ことを示している。

失ったものははっきりしている。`1/144`(スケール = ホビー)、`26cm`(サイズ = ファッション)、
`xr-500b`(型番 = 家電)は、いずれも**カテゴリを名指しできる強い手がかり**だった。
概念4 の ⑤ で見た係数の top8 に `144` や `27 cm` が並んでいたことを思い出そう。あれを自分で捨てたことになる。

> 教訓: **正規化は「揃える」ためにやるのであって、「消す」ためにやるのではない。**
> 表記が違うだけの別物を1つにするのは常に得。意味を持つ違いまで潰し始めた瞬間に損に変わる。
> そしてその境界がどこかは、データを見ただけでは分からない。**測って決める。**

## 答え合わせ: public LB と private LB

コンペ中は `test` の正解が見えない。だが今回は**教材なので、最後に答えを見せる**。

競技本番と同様、この教材も `test` の正解カテゴリは配布しない。次のセルは、明示的に管理者用の正解ファイルを配置した場合だけ採点する。
`test` を先頭30%(= public)と残り70%(= private)に分けて、それぞれ macro-F1 を出す。

見るポイントは3つ:

1. **モデルなし(常に最多クラス)→ 素のテキスト → 正規化あり** でスコアがどう動いたか
2. **モデルなしの提出は accuracy がそこそこでも macro-F1 が壊滅する**こと。
   指標の選び方ひとつで「良いモデル」の定義が変わる
3. **public と private でスコアが一致しない**こと。件数が少ないほど揺れる

> 概念3の ⑦ を書いていれば、君の作った行列を使った提出も一緒に採点される。
> 未記入なら 0〜2 番だけが出る(notebook は止まらない)。

In [ ]:
# GOAL: 自分のベースラインが test 上で何点だったのかを見る(コンペ終了後の答え合わせという設定)

ANSWER = DATA / "_competition_answers_not_distributed.csv"

if not ANSWER.exists():
    print("答えファイルが見つかりません(このセルはスキップして構いません):", ANSWER)
else:
    answer = pd.read_csv(ANSWER)
    y_test = answer.set_index("listing_id").loc[test["listing_id"], "category"].to_numpy()
    n = len(y_test)
    k = int(n * 0.3)   # 先頭30% = public LB、残り70% = private LB という設定

    entries = []

    # 0. モデルなし: 常に最多クラスと答える
    major = pd.Series(y).value_counts().idxmax()
    entries.append(("0. 常に『" + str(major) + "』と答える", np.array([major] * n)))

    # 1. 正規化なし + TF-IDF + ロジスティック回帰
    m_raw = make_model().fit(RAWDOCS_TRAIN, y)
    entries.append(("1. 素のテキスト", m_raw.predict(RAWDOCS_TEST)))

    # 2. 正規化あり(このレッスンの成果物)
    m_norm = make_model().fit(DOCS_TRAIN, y)
    entries.append(("2. 正規化してから", m_norm.predict(DOCS_TEST)))

    # 3. ⑦(概念3)を書いていれば、その行列を使った版も出す
    _vec, _xtr, _xte = globals().get("vec"), globals().get("X_tr"), globals().get("X_te")
    if _xtr is not None and _xte is not None and sp_info(_xtr, "cols") == sp_info(_xte, "cols"):
        clf7 = LogisticRegression(C=4.0, max_iter=2000).fit(_xtr, y)
        entries.append(("3. 君の ⑦ の行列で", clf7.predict(_xte)))
    else:
        print("(概念3の ⑦ が未記入なので 3 はスキップ。書いてから再実行しよう)\n")

    print(f"{'提出':<30}{'public F1':>11}{'private F1':>12}{'private acc':>13}")
    print("-" * 66)
    for label, p in entries:
        pub = f1_score(y_test[:k], p[:k], average="macro", zero_division=0)
        pri = f1_score(y_test[k:], p[k:], average="macro", zero_division=0)
        pacc = accuracy_score(y_test[k:], p[k:])
        print(f"{label:<26}{pub:>11.4f}{pri:>12.4f}{pacc:>13.4f}")

    print("\n※ macro-F1 は大きいほど良い。0番(モデルなし)は accuracy はそこそこでも macro-F1 が壊滅する。")
    print("※ public(300件)と private(700件)で数値が違う。件数が少ないほど揺れる。")
    print("※ ここまで使ったモデルは線形1本。効いているのは前処理とベクトル化の設計。")

---
## 振り返り(自己評価 + TIL)

以下に**1〜2文ずつ**、自分の言葉で書いてみよう。書いた内容はセッション終了時の学習ノートと
スキルレベルの判定に使う(空欄でも先に進めるが、言語化すると定着が大きく変わる)。

**1. 今日学んだことを自分の言葉で:**

> (ここに書く)

**2. 難しかったこと・まだあやふやなこと:**

> (ここに書く)

**3. Feynman チェック — 次の3つに、資料を見ずに答えられる?**

- 「なぜ正規化すると語彙が減るのに精度が上がるのか」を、語彙表(`Dictionary<string,int>`)の言葉で説明できる?
- 「test に `fit_transform` を使うと何が2つ同時に壊れるのか」を言える?
- 「accuracy 0.94 のモデルです」という報告に対して、**追加で何を確認すべきか**を言える?

> (ここに書く)

---
## まとめ

### 今日学んだこと

| # | 概念 | 一言でいうと |
|---|---|---|
| 1 | 語彙表という見方 | モデルにとって語は `Dictionary<string,int>` のキー。**表記がブレると手がかりが割れて薄まる** |
| 2 | NFKC | `unicodedata.normalize("NFKC", s)` で半角カナ・全角英数・全角記号を1つの代表形に潰す。ひらがな/カタカナは統一しない |
| 3 | HTML 残渣 | `html.unescape` → タグ除去の順。エスケープされたタグがあるので unescape が先 |
| 4 | テンプレ文言 | 全文書共通なら情報ゼロ。サイトごとに共通する文言なら**収集元を当てる手がかり = リーク源**になる |
| 5 | プレースホルダ化 | 価格の4通りの書き方を ` <num> ` 1トークンに潰す。桁は `price` 列が持っている |
| 6 | 正規化のやりすぎ | 数字を全部潰すと型番 `XR-500B` やサイズ `26cm` まで死ぬ。**境界はスコアで決める** |
| 7 | 正規化の順番 | unescape → タグ → NFKC → 装飾 → テンプレ → 数値 → 小文字 → **空白畳み込みは必ず最後** |
| 8 | 形態素解析 | 日本語には区切りが無い。janome が辞書 + 最短経路探索で切る。`surface` / `base_form` / `part_of_speech` |
| 9 | `Tokenizer()` は1個 | 辞書ロードが重い。ループ内で作ると数百倍遅くなる |
| 10 | 分かち書き文字列 | `"東京 都 の 中古 テレビ"` の形にして、英語向けの道具をそのまま使う |
| 11 | BoW | 語彙表 + 出現回数。語順は捨てる。**99% がゼロ = 疎行列** |
| 12 | TF-IDF | tf(その文書での多さ)× idf(全体での珍しさ)。行は L2 正規化される |
| 13 | `ngram_range` | (1,2) で「連続2語」も列に。語彙は数倍に膨らむ |
| 14 | `min_df` / `max_df` | 珍しすぎる語・ありふれすぎる語を落として語彙を絞る |
| 15 | `sublinear_tf` | tf を `1+log(tf)` に。10回出た語を10倍重くしない |
| 16 | 疎行列(CSR) | `.shape` / `.nnz`。**`.toarray()` はメモリ爆発**の入り口 |
| 17 | `fit` は train だけ | test は `transform` だけ。破ると列不一致 + 語彙リークの二重事故 |
| 18 | Pipeline | 前処理とモデルを1部品に。CV で fold ごとの fit が**構造的に**守られる |
| 19 | accuracy vs macro-F1 | 不均衡なら macro-F1。accuracy は多数クラスの成績にほぼ支配される |
| 20 | `classification_report` | precision(誤検知の少なさ)/ recall(取りこぼしの少なさ)/ support(揺れやすさの目安) |
| 21 | `class_weight="balanced"` | 少数クラスの誤りを重く罰する。accuracy とのトレードオフを測って選ぶ |
| 22 | 係数を読む | `coef_` は (クラス数, 語彙数)。**上位語を見れば前処理の漏れが分かる** |

### この先どこで使うか(先読み)

- **unit06(あいまい文字列マッチと名寄せ)** — 今日の正規化は、そのまま**名寄せの前処理**になる。
  「同じ商品か」を判定する前に表記を揃えるのは同じ発想で、そこに編集距離(競プロの DP がそのまま出る)が乗る。
  今日の `wakati` も、文字 n-gram との比較材料として再登場する。
- **unit07(埋め込みと意味検索)** — 今日の TF-IDF は「語が一致しないと類似度ゼロ」という弱点を持つ。
  `中古` と `ユーズド` が別物になる問題を、密ベクトル(埋め込み)がどう解くかに進む。
  **TF-IDF ベースラインを今日作ったからこそ、埋め込みの価値を数字で語れる。**
- **unit09(モデル選択)** — 今日の「線形ベースラインを先に置く」姿勢がそのまま主題になる。
  BERT 系を使う判断は、今日のスコアとの差分で下す。
- **unit10(本番運用)** — 前処理関数は **train と推論で完全に同じものを使う**必要がある。
  今日の `normalize_base` のような関数を1箇所に固めて共有するのが、本番でのバグを防ぐ唯一の方法だ。
- **実務(Web データ活用)** — 収集したテキストを分類・検索・名寄せするとき、
  今日の正規化パイプラインは**そのまま流用できる資産**になる。

### 次にやること

**演習 `ex01_clean_and_normalize` へ進もう。lesson.ipynb を見ながらで OK。**
思い出せない API があれば ② の表に戻ればいい。暗記ではなく、**どこを見れば分かるか**を覚えているのが実務の状態だ。

演習は4本:

| 演習 | 内容 |
|---|---|
| `ex01_clean_and_normalize` | 正規化パイプラインを関数にまとめ、汚れの残存をテストで潰す |
| `ex02_tokenize_and_tfidf` | janome の分かち書き + TF-IDF ベクトル化(fit/transform の分離をテストで強制) |
| `ex03_linear_baseline` | 線形モデルで CV を回し、macro-F1 で評価する |
| `ex04_capstone` | 文字 n-gram Pipeline と OOF 分類を一気通貫 |